<a href="https://colab.research.google.com/github/ArshnoorSinghh/ML-Project/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

# read just the March 2026 partition (one month = small enough for pandas)
url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03"
df = pd.read_parquet(url, storage_options={"token": HF_TOKEN})

print(df.shape)
df.head()

(9841378, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

My contract (Lane 4):
1. One row = one page (content item) on one day.
2. Table: fact_content_daily_performance (daily), joined later with dim_content for page info.
3. Time window: develop on month=2026-03; final month (June 2026) is a sealed test.
4. What I rank: the CTR gap — expected CTR for a page's position minus its actual CTR. Bigger gap = under-earning.
5. Deliberately excluded: any future-window click/impression data that overlaps the outcome I score, since using it would leak the answer.

In [3]:
len(df)

9841378

In [4]:
df["report_date"].nunique()

31

In [5]:
df["content_hash_id"].nunique()

331437

In [6]:
df.groupby(["report_date","content_hash_id"]).ngroups

9841378

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [7]:
print("Total rows:", len(df))
print("Earliest date:", df["report_date"].min())
print("Latest date:", df["report_date"].max())
print("Number of distinct pages:", df["content_hash_id"].nunique())

Total rows: 9841378
Earliest date: 2026-03-01
Latest date: 2026-03-31
Number of distinct pages: 331437


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Five features, each knowable at the decision moment:
- gsc_impressions: past display count up to the decision date — knowable.
- gsc_clicks: past clicks already observed — knowable.
- ctr: computed from past clicks/impressions — knowable.
- avg_position: observed average rank so far — knowable.
- (position tier from avg_position): derived, no future info — knowable.

In [8]:
available = df[df["gsc_data_available"] == True]
print("Rows total:      ", len(df))
print("Rows with GSC:   ", len(available))
print("Share available: ", round(len(available)/len(df)*100, 1), "%")

Rows total:       9841378
Rows with GSC:    3611061
Share available:  36.7 %


In [9]:
work = df[df["gsc_data_available"] == True].copy()
work["ctr"] = work["gsc_clicks"] / work["gsc_impressions"]
work["avg_position"] = work["gsc_sum_position"] / work["gsc_impressions"]

features = work[["content_hash_id", "gsc_impressions", "gsc_clicks", "ctr", "avg_position"]].copy()
features.head()

,content_hash_id,gsc_impressions,gsc_clicks,ctr,avg_position
0,content_b7e512995f79d5a6,20,0,0.000,3.350000
1,content_05597932fe4da067,1,0,0.000,0.000000
2,content_7a105f548d9c6916,125,1,0.008,4.928000
3,content_905aa32a0230694e,7,0,0.000,4.000000
4,content_a3ea9792f793ec72,11,0,0.000,2.272727


In [10]:
# LEAKAGE TRAP (on purpose) -----------------------------------------
# Build the target: CTR gap = expected CTR for a position tier minus actual CTR.
work["position_tier"] = pd.cut(work["avg_position"], [0,3,10,20,1000],
                               labels=["top_3","page_1","page_2","deep"])
work["expected_ctr"] = work.groupby("position_tier", observed=True)["ctr"].transform("median")
work["ctr_gap"] = work["expected_ctr"] - work["ctr"]      # this is the TARGET

# A HONEST feature (impressions) vs the TARGET:
honest_corr = work["gsc_impressions"].corr(work["ctr_gap"])

# A LEAKY feature: derived straight from the target itself.
work["leaky_feature"] = work["ctr_gap"] * 1.0             # basically the answer
leaky_corr = work["leaky_feature"].corr(work["ctr_gap"])

print("Honest feature (impressions) correlation with target:", round(honest_corr, 3))
print("Leaky feature correlation with target:              ", round(leaky_corr, 3))
print("-> leaky = perfect 1.0 because it IS the answer in disguise. Deleting it.")

# Delete the leak, keep the honest setup.
work = work.drop(columns=["leaky_feature"])
print("Leaky column removed. Honest features remain.")

Honest feature (impressions) correlation with target: 0.001
Leaky feature correlation with target:               1.0
-> leaky = perfect 1.0 because it IS the answer in disguise. Deleting it.
Leaky column removed. Honest features remain.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data limits of my slice:
- Only 36.7% of March rows have GSC search data available, so my Lane 4
  work is restricted to that subset — the rest have no CTR to score.
- The panel is unbalanced: clients started tracking at different dates
  (see dim_clients gsc_data_start), so history depth varies by client.
- This month alone can't tell me a page's future trend — that needs
  windows across months, and the feature/target windows must never overlap
  or the score leaks (as the trap above showed).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.